<a href="https://colab.research.google.com/github/uin33273/GoogleColab_practice/blob/main/%E7%AE%97%E5%AE%9A%E5%9F%BA%E6%BA%962_%EF%BC%91%E3%81%A4%E3%81%AE%E3%82%B7%E3%83%BC%E3%83%88%E3%81%A8spreadsheet%E9%80%A3%E7%B5%90.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import pandas as pd
import openpyxl
from openpyxl.utils import get_column_letter
from google.colab import files
import io
from ipywidgets import widgets
from IPython.display import display, clear_output
import re

# --- 1. ファイルアップロード ---
print("【手順】")
print("1. 『算定区分・加算等集計表』をExcelでダウンロードしてください。")
print("2. 以下の順にファイルをアップロードしてください。")

print("\n--- 1つ目：複数を1ファイルに.csv ---")
up_csv = files.upload()
csv_filename = list(up_csv.keys())[0]
df_csv = pd.read_csv(io.BytesIO(up_csv[csv_filename]), encoding='cp932')

print("\n--- 2つ目：算定区分・加算等集計表.xlsx ---")
up_xlsx = files.upload()
xlsx_filename = list(up_xlsx.keys())[0]
df_xlsx = pd.read_excel(io.BytesIO(up_xlsx[xlsx_filename]), sheet_name='集計')

# --- 2. 準備 ---
csv_shops = df_csv.iloc[:, 1].astype(str)
xlsx_master = sorted(df_xlsx.iloc[:, 3].dropna().astype(str).unique().tolist())

while len(df_csv.columns) < 11:
    df_csv[f'Extra_{len(df_csv.columns)}'] = ""
k_idx = 10

EXCLUDE_WORDS = ["店", "店舗", "支店"]

def get_clean_place(text, is_csv=False):
    target = text.split('_')[0] if is_csv and '_' in text else text
    clean = re.sub(r'[0-9a-zA-Z!-~ 　\(\)（）/／]+', '', target)
    clean = clean.replace("メソッド", "")
    for word in EXCLUDE_WORDS:
        clean = clean.replace(word, "")
    return clean

# --- 3. 照合ロジック ---
pending_list = []
output_filename = "2-spreadsheetと複数から１つへ.csv"
print("\n照合中...")

for idx in range(len(df_csv)):
    raw_name = str(csv_shops[idx])
    target_place = get_clean_place(raw_name, is_csv=True)
    place_matches = [m for m in xlsx_master if target_place == get_clean_place(m)]

    # 「／」が含まれる場合、地名一致分を表示。なければ全表示
    if "／" in raw_name or "/" in raw_name:
        display_options = place_matches if place_matches else xlsx_master
        pending_list.append({'idx': idx, 'raw': raw_name, 'matches': display_options})
        continue

    # 通常判定
    is_p_case = "児" in raw_name or "P" in raw_name.upper()
    final_matches = [m for m in place_matches if ("P" if is_p_case else "M") in m.upper()]

    if len(final_matches) == 1:
        df_csv.iloc[idx, k_idx] = final_matches[0]
    else:
        # 候補複数 or 0件
        display_options = final_matches if final_matches else place_matches if place_matches else xlsx_master
        pending_list.append({'idx': idx, 'raw': raw_name, 'matches': display_options})

# --- 4. UI表示（絞り込み機能付き Combobox） ---
def on_save_clicked(b):
    df_csv.to_csv(output_filename, index=False, encoding='cp932')
    files.download(output_filename)
    print(f"\n【{output_filename}】のダウンロードを完了しました。")

save_btn = widgets.Button(description="保存してダウンロード", button_style='success', layout={'width': '250px'})
save_btn.on_click(on_save_clicked)

if pending_list:
    print(f"\n【手動修正】入力欄に文字を入れると候補が絞り込まれます。")
    for item in pending_list:
        idx = item['idx']

        # Dropdownの代わりにComboboxを使用
        combo = widgets.Combobox(
            options=item['matches'],
            placeholder='店名を入力して検索...',
            description=f"行{idx+1}:",
            ensure_option=True, # リストにあるもののみ選択可能
            style={'description_width': 'initial'},
            layout={'width': '500px'}
        )

        def make_update(row_idx):
            def _update(change):
                if change['new']:
                    df_csv.iloc[row_idx, k_idx] = change['new']
            return _update

        combo.observe(make_update(idx), names='value')
        print(f"CSV店名: {item['raw']}")
        display(combo)
else:
    print("\nすべて自動判別しました。")

print("\n最後に以下のボタンを押して完了してください。")
display(save_btn)

【手順】
1. 『算定区分・加算等集計表』をExcelでダウンロードしてください。
2. 以下の順にファイルをアップロードしてください。

--- 1つ目：複数を1ファイルに.csv ---


Saving 1-複数を1ファイルに.csv to 1-複数を1ファイルに (4).csv

--- 2つ目：算定区分・加算等集計表.xlsx ---


Saving 算定区分・加算等集計表（運営用） (1).xlsx to 算定区分・加算等集計表（運営用） (1) (4).xlsx

照合中...

【手動修正】入力欄に文字を入れると候補が絞り込まれます。
CSV店名: パーク阿見P_2025年11月_集計一覧.csv


Combobox(value='', description='行3:', ensure_option=True, layout=Layout(width='500px'), options=('001M岩曽', '00…

CSV店名: 筑西プラスP／筑西プラス_2025年11月_集計一覧 放.csv


...

<IPython.core.display.Javascript object>


【2-spreadsheetと複数から１つへ.csv】のダウンロードを完了しました。


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


【2-spreadsheetと複数から１つへ.csv】のダウンロードを完了しました。
